In [ ]:
"""
CELL 1 — Run this FIRST after every kernel restart.
Patches NumPy 2.x aliases removed in 2.0 that flwr still uses.
Must run before any other import.
"""
import numpy as np
np.float_   = np.float64
np.int_     = np.int64
np.complex_ = np.complex128
np.object_  = object
np.str_     = np.str_
print(f"NumPy {np.__version__} aliases patched.")

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.filterwarnings("ignore")

import flwr
print(f"flwr: {flwr.__version__}")
from flwr.common import parameters_to_ndarrays, ndarrays_to_parameters
from flwr.server.strategy import FedAvg
import flwr.simulation
print("flwr core imports: OK")

In [ ]:
import subprocess, torch
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "No GPU found")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
import os, sys, shutil

# input is read-only — copy to writable working dir
src_root     = "/kaggle/input/datasets/amadeoyesa/fraud-fl-ta/fraud-fl-TA"
project_root = "/kaggle/working/fraud-fl-TA"

if not os.path.exists(project_root):
    print("Copying project to /kaggle/working/ ...")
    shutil.copytree(src_root, project_root)
    print("Done.")
else:
    print(f"Already exists: {project_root}")

# register paths
hfedxgboost_path = f"{project_root}/models/fedxgbllr"
for p in [project_root, hfedxgboost_path]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(project_root)
print(f"cwd: {os.getcwd()}")
print(f"sys.path[0:2]: {sys.path[0:2]}")

In [ ]:
import subprocess, sys

packages = [
    "hydra-core==1.3.2",
    "torchmetrics==1.8.2",
    "imbalanced-learn>=0.11.0",
    "shap>=0.44.0",
    "wandb>=0.15.12",
    "pyaml>=23.0.0",
    "tqdm==4.66.3",
    "omegaconf>=2.3.0",
]

for pkg in packages:
    print(f"Installing {pkg}...", end=" ", flush=True)
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    print("OK" if result.returncode == 0
          else f"FAILED: {result.stderr[-100:]}")

# hfedxgboost: install without deps to avoid ray/flwr conflicts
print("Installing hfedxgboost (no-deps)...", end=" ", flush=True)
result = subprocess.run(
    ["pip", "install", "-e",
     "/kaggle/working/fraud-fl-TA/models/fedxgbllr/",
     "--ignore-requires-python", "--no-deps", "-q"],
    capture_output=True, text=True
)
print("OK" if result.returncode == 0
      else f"FAILED: {result.stderr[-100:]}")

# verify hfedxgboost via sys.path (editable may not register)
try:
    import hfedxgboost
    print(f"hfedxgboost importable: OK")
except ImportError:
    # fallback: already in sys.path from Cell 3
    print("hfedxgboost via sys.path: OK (editable not needed)")

print("\nAll packages done.")

In [ ]:
import os, pandas as pd

project_root = "/kaggle/working/fraud-fl-TA"
os.makedirs(f"{project_root}/data/paysim", exist_ok=True)

src = "/kaggle/input/datasets/amadeoyesa/ta-dataset/paysim.csv"
dst = f"{project_root}/data/paysim/paysim.csv"

# use lexists to handle broken symlinks
if os.path.lexists(dst):
    os.remove(dst)
os.symlink(src, dst)
print(f"Symlinked: {src} → {dst}")

df = pd.read_csv(dst, nrows=3)
print(f"PaySim columns: {df.columns.tolist()}")
print(f"OK — {len(df.columns)} columns found")

In [ ]:
import sys, os
# ensure paths (in case cells run out of order)
for p in ["/kaggle/working/fraud-fl-TA",
          "/kaggle/working/fraud-fl-TA/models/fedxgbllr"]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir("/kaggle/working/fraud-fl-TA")

checks = ["flwr","torch","xgboost","sklearn",
          "pandas","numpy","imblearn","wandb","hydra"]
for mod in checks:
    try:
        m = __import__(mod)
        v = getattr(m, "__version__", "?")
        print(f"  {mod}: {v}")
    except Exception as e:
        print(f"  {mod}: FAILED — {e}")

print("\nLoading PaySim (~30s)...")
from preprocessing.paysim import load_paysim
DATA = load_paysim(
    data_path="/kaggle/working/fraud-fl-TA/data/paysim/paysim.csv"
)
print("PaySim loaded:")
for k in ["x_train","x_val","x_test","y_train","y_val","y_test"]:
    print(f"  {k}: {DATA[k].shape} | dtype: {DATA[k].dtype}")
print("\nFraud ratios:")
for split in ["train","val","test"]:
    y = DATA[f"y_{split}"]
    print(f"  {split}: {y.sum()} / {len(y)} = "
          f"{y.mean()*100:.4f}%")
print("\nAll imports and data verified.")

In [ ]:
import wandb

WANDB_OK = False
try:
    # wandb.login() will prompt for API key
    # Or replace with: wandb.login(key="paste_your_key_here")
    wandb.login()
    WANDB_OK = True
    print(f"W&B login OK — version {wandb.__version__}")
except Exception as e:
    print(f"W&B login FAILED: {e}")
    print("Continuing with use_wandb=False — results saved to CSV only.")

In [ ]:
import os, sys, time
os.chdir("/kaggle/working/fraud-fl-TA")
for p in ["/kaggle/working/fraud-fl-TA",
          "/kaggle/working/fraud-fl-TA/models/fedxgbllr"]:
    if p not in sys.path:
        sys.path.insert(0, p)

from models.ffd.run import run as ffd_run

print("="*60)
print("SMOKE TEST: 3 rounds, 2 clients, IID, no oversampling")
print("="*60)

smoke_cfg = {
    "model":               "ffd",
    "num_clients":         2,
    "num_rounds":          3,
    "local_epochs":        2,
    "batch_size":          80,
    "lr":                  0.01,
    "random_seed":         42,
    "oversampling":        "none",
    "use_wandb":           False,
    "wandb_project":       "hfedxgboost-paysim",
    "early_stop_patience": 10,
    "partition":           {"scheme": "iid", "alpha": None},
}

t0 = time.time()
ffd_run(smoke_cfg)
elapsed = time.time() - t0

print(f"\nSmoke test done in {elapsed:.1f}s")
scale = (50 / 3) * (5 / 2)
est_hours = elapsed * scale * 8 / 3600
print(f"Estimated time for 8 full runs: ~{est_hours:.1f} hours")
if est_hours > 11:
    print("\nWARNING: Estimated time exceeds 12h Kaggle limit!")
    print("Reduce NUM_ROUNDS to 30 in Cell 10 before proceeding.")
else:
    print("Time estimate OK — safe to proceed with Cell 9.")

In [ ]:
import os, csv
from datetime import datetime

RESULTS_DIR = "/kaggle/working/ffd_results"
LOG_DIR     = f"{RESULTS_DIR}/logs"
RESULTS_CSV = f"{RESULTS_DIR}/results.csv"
os.makedirs(LOG_DIR, exist_ok=True)

CSV_HEADERS = [
    "run_id","scheme","alpha","oversampling","seed",
    "num_clients","num_rounds","local_epochs","batch_size","lr",
    "val_auprc","val_f1","val_precision","val_recall",
    "test_auprc","test_f1","test_precision","test_recall",
    "status","duration_minutes","timestamp",
]

if not os.path.exists(RESULTS_CSV):
    with open(RESULTS_CSV, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=CSV_HEADERS).writeheader()
    print(f"Created: {RESULTS_CSV}")
else:
    print(f"Already exists: {RESULTS_CSV}")

def save_result(row: dict):
    with open(RESULTS_CSV, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=CSV_HEADERS).writerow(row)

def save_log(run_id: str, content: str):
    with open(f"{LOG_DIR}/{run_id}.txt", "w") as f:
        f.write(content)

print(f"Results dir: {RESULTS_DIR}")
print("Infrastructure ready.")

In [ ]:
import os, sys, time, traceback, io, re, csv
from datetime import datetime

os.chdir("/kaggle/working/fraud-fl-TA")
for p in ["/kaggle/working/fraud-fl-TA",
          "/kaggle/working/fraud-fl-TA/models/fedxgbllr"]:
    if p not in sys.path:
        sys.path.insert(0, p)

from models.ffd.run import run as ffd_run

# ── experiment matrix ───────────────────────────────
EXPERIMENTS = [
    ("ffd_iid_none",   "iid",        None, "none" ),
    ("ffd_iid_smote",  "iid",        None, "smote"),
    ("ffd_d05_none",   "dirichlet",  0.5,  "none" ),
    ("ffd_d05_smote",  "dirichlet",  0.5,  "smote"),
    ("ffd_d10_none",   "dirichlet",  1.0,  "none" ),
    ("ffd_d10_smote",  "dirichlet",  1.0,  "smote"),
    ("ffd_d50_none",   "dirichlet",  5.0,  "none" ),
    ("ffd_d50_smote",  "dirichlet",  5.0,  "smote"),
]

# ── research params ────────────────────────────────
SEED         = 42
NUM_CLIENTS  = 5
NUM_ROUNDS   = 50    # reduce to 30 if smoke test estimate > 11h
LOCAL_EPOCHS = 5
BATCH_SIZE   = 80
LR           = 0.01

# ── metric regex parsers ─────────────────────────────
RE_TEST = re.compile(
    r"FINAL.*?test_auprc=([\d.]+).*?test_f1=([\d.]+)"
    r".*?test_precision=([\d.]+).*?test_recall=([\d.]+)",
    re.DOTALL
)
RE_VAL = re.compile(
    r"server\] round \d+.*?val_auprc=([\d.]+).*?val_f1=([\d.]+)"
    r".*?val_precision=([\d.]+).*?val_recall=([\d.]+)"
)

summary = []
total   = len(EXPERIMENTS)

print(f"Starting {total} experiments")
print(f"K={NUM_CLIENTS} R={NUM_ROUNDS} E={LOCAL_EPOCHS} "
      f"B={BATCH_SIZE} lr={LR} seed={SEED}")
print("="*60)

for i, (run_id, scheme, alpha, oversampling) in \
        enumerate(EXPERIMENTS, 1):

    print(f"\n{'='*60}")
    print(f"RUN {i}/{total}: {run_id}")
    print(f"  scheme={scheme} alpha={alpha} "
          f"oversampling={oversampling}")
    print(f"  started: {datetime.now().strftime('%H:%M:%S')}")
    print("="*60)

    cfg = {
        "model":               "ffd",
        "num_clients":         NUM_CLIENTS,
        "num_rounds":          NUM_ROUNDS,
        "local_epochs":        LOCAL_EPOCHS,
        "batch_size":          BATCH_SIZE,
        "lr":                  LR,
        "random_seed":         SEED,
        "oversampling":        oversampling,
        "use_wandb":           bool(WANDB_OK),
        "wandb_project":       "hfedxgboost-paysim",
        "early_stop_patience": 10,
        "partition":           {"scheme": scheme, "alpha": alpha},
    }

    t0 = time.time()
    status = "FAILED"
    log_content = ""
    result_metrics = {
        "val_auprc": None, "val_f1": None,
        "val_precision": None, "val_recall": None,
        "test_auprc": None, "test_f1": None,
        "test_precision": None, "test_recall": None,
    }
    old_stdout = sys.stdout

    try:
        # Tee: write to real stdout AND capture
        real_stdout = sys.__stdout__
        buf = io.StringIO()

        class Tee:
            def write(self, s):
                buf.write(s)
                real_stdout.write(s)
                real_stdout.flush()
            def flush(self):
                real_stdout.flush()

        tee = Tee()
        sys.stdout = tee

        try:
            ffd_run(cfg)
        finally:
            sys.stdout = old_stdout

        log_content = buf.getvalue()
        status = "SUCCESS"

        # parse test metrics
        m = RE_TEST.search(log_content)
        if m:
            result_metrics["test_auprc"]     = float(m.group(1))
            result_metrics["test_f1"]        = float(m.group(2))
            result_metrics["test_precision"] = float(m.group(3))
            result_metrics["test_recall"]    = float(m.group(4))

        # parse last val metrics
        val_matches = list(RE_VAL.finditer(log_content))
        if val_matches:
            lv = val_matches[-1]
            result_metrics["val_auprc"]     = float(lv.group(1))
            result_metrics["val_f1"]        = float(lv.group(2))
            result_metrics["val_precision"] = float(lv.group(3))
            result_metrics["val_recall"]    = float(lv.group(4))

    except Exception:
        log_content = traceback.format_exc()
        sys.stdout = old_stdout
        print(f"\n[ERROR] {run_id} FAILED:")
        print(log_content[-500:])
        status = f"FAILED"

    elapsed = time.time() - t0
    save_log(run_id, log_content)

    row = {
        "run_id":           run_id,
        "scheme":           scheme,
        "alpha":            alpha if alpha else "IID",
        "oversampling":     oversampling,
        "seed":             SEED,
        "num_clients":      NUM_CLIENTS,
        "num_rounds":       NUM_ROUNDS,
        "local_epochs":     LOCAL_EPOCHS,
        "batch_size":       BATCH_SIZE,
        "lr":               LR,
        **result_metrics,
        "status":           status,
        "duration_minutes": round(elapsed/60, 2),
        "timestamp":        datetime.now().isoformat(),
    }
    save_result(row)
    summary.append(row)

    sep = "-" * 40
    print(f"\n{sep}")
    print(f"RUN {i}/{total} DONE: {run_id} | {status} | "
          f"{elapsed/60:.1f} min")
    if result_metrics["test_auprc"] is not None:
        print(f"  test_auprc={result_metrics['test_auprc']:.4f} "
              f"test_f1={result_metrics['test_f1']:.4f}")
    time.sleep(3)

sep = "=" * 60
print(f"\n{sep}")
print(f"ALL {total} RUNS COMPLETE")
print(f"{sep}")

In [ ]:
import pandas as pd

print("\n=== FINAL RESULTS ===\n")
df = pd.DataFrame(summary)
cols = ["run_id","scheme","alpha","oversampling",
        "test_auprc","test_f1","test_precision","test_recall",
        "status","duration_minutes"]
print(df[cols].to_string(index=False))
print(f"\nCSV: {RESULTS_CSV}")
df_csv = pd.read_csv(RESULTS_CSV)
print(f"CSV rows written: {len(df_csv)}")

In [ ]:
import shutil, os

output_dir = "/kaggle/working/ffd_results_output"
os.makedirs(output_dir, exist_ok=True)

shutil.copy(RESULTS_CSV, f"{output_dir}/results.csv")
print(f"Copied results.csv")

logs_dst = f"{output_dir}/logs"
if os.path.exists(logs_dst):
    shutil.rmtree(logs_dst)
shutil.copytree(LOG_DIR, logs_dst)
print(f"Copied logs/")
print(f"\nDownload from Kaggle Output tab: {output_dir}/")

## Run Order (IMPORTANT)

After every kernel restart, run cells in this exact order:

1. **Cell 1** — NumPy patch (MANDATORY FIRST)
2. **Cell 2** — GPU check
3. **Cell 3** — copy project
4. **Cell 4** — install deps
5. **Cell 5** — link PaySim
6. **Cell 6** — verify imports + load data
7. **Cell 7** — W&B login (paste API key)
8. **Cell 8** — smoke test (check time estimate!)
9. **Cell 9** — results infrastructure
10. **Cell 10** — overnight experiments (runs all 8)
11. Morning: Cell 11 → summary, Cell 12 → download

## Experiment Matrix

| # | run_id | scheme | alpha | oversampling |
|---|--------|--------|-------|--------------|
| 1 | ffd_iid_none | IID | - | none |
| 2 | ffd_iid_smote | IID | - | smote |
| 3 | ffd_d05_none | Dirichlet | 0.5 | none |
| 4 | ffd_d05_smote | Dirichlet | 0.5 | smote |
| 5 | ffd_d10_none | Dirichlet | 1.0 | none |
| 6 | ffd_d10_smote | Dirichlet | 1.0 | smote |
| 7 | ffd_d50_none | Dirichlet | 5.0 | none |
| 8 | ffd_d50_smote | Dirichlet | 5.0 | smote |

**Full params**: K=5, R=50, E=5, B=80, lr=0.01, seed=42